In [ ]:
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb || apt-get install -fy
!pip install selenium beautifulsoup4 pandas openpyxl webdriver-manager

Selecting previously unselected package google-chrome-stable.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack google-chrome-stable_current_amd64.deb ...
Unpacking google-chrome-stable (157.0.8103.38-1) ...
dpkg: dependency problems prevent configuration of google-chrome-stable:
 google-chrome-stable depends on libatk-bridge2.0-0 (>= 2.5.3); however:
  Package libatk-bridge2.0-0 is not installed.
 google-chrome-stable depends on libatk1.0-0 (>= 2.11.90); however:
  Package libatk1.0-0 is not installed.
 google-chrome-stable depends on libatspi2.0-0 (>= 2.9.90); however:
  Package libatspi2.0-0 is not installed.
 google-chrome-stable depends on libxcomposite1 (>= 1:0.4.4-1); however:
  Package libxcomposite1 is not installed.

dpkg: error processing package google-chrome-stable (--install):
 dependency problems - leaving unconfigured
Processing triggers for man-db (2.12.0-4build2) ...
Errors were encountered while processing:
 google-chrome-st

In [7]:
# JulioPrajaSarbunan_202431152
import time
import re
import csv
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

In [8]:
kata_kunci = ['kaos pria', 'kemeja pria', 'jaket pria', 'polo shirt pria', 'hoodie pria']
max_halaman = 5

chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--disable-gpu')
chrome_options.add_argument('--window-size=1920,1080')
chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36')

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

def baca_kartu(teks):
    p = [x.strip() for x in teks.split('|')]
    harga_idx = [i for i, x in enumerate(p) if x.startswith('Rp')]
    if not harga_idx:
        return None
    i = harga_idx[0]
    nama = p[i - 1] if i > 0 else '-'
    harga = p[i]
    harga_asli = p[harga_idx[1]] if len(harga_idx) > 1 else harga
    rating = next((x for x in p if re.fullmatch(r'[0-5]\.\d', x)), '-')
    terjual = next((x for x in p if 'terjual' in x.lower()), '-')
    toko = p[-2] if len(p) >= 2 else '-'
    lokasi = p[-1]
    return {
        'nama': nama,
        'harga': harga,
        'harga_asli': harga_asli,
        'rating': rating,
        'terjual': terjual,
        'toko': toko,
        'lokasi': lokasi
    }

result = []
sudah = set()

try:
    for kw in kata_kunci:
        for halaman in range(1, max_halaman + 1):
            url = f"https://www.tokopedia.com/search?q={kw.replace(' ', '+')}&page={halaman}"
            driver.get(url)
            time.sleep(5)

            for i in range(15):
                driver.execute_script("window.scrollBy(0, 600);")
                time.sleep(1)

            soup = BeautifulSoup(driver.page_source, "html.parser")
            baru = 0
            for a in soup.find_all('a', href=True):
                teks = a.get_text('|', strip=True)
                if 'Rp' not in teks or 'terjual' not in teks:
                    continue
                kunci = a['href'].split('?')[0]
                if kunci in sudah:
                    continue
                row = baca_kartu(teks)
                if row is None:
                    continue
                row['produk'] = a['href']
                sudah.add(kunci)
                result.append(row)
                baru += 1

            print(f"{kw} halaman {halaman}: {baru} data baru, total {len(result)}")
            if baru == 0:
                break
finally:
    driver.quit()

print(len(result))

kaos pria halaman 1: 60 data baru, total 60
kaos pria halaman 2: 0 data baru, total 60
kemeja pria halaman 1: 0 data baru, total 60
jaket pria halaman 1: 179 data baru, total 239
jaket pria halaman 2: 0 data baru, total 239
polo shirt pria halaman 1: 0 data baru, total 239
hoodie pria halaman 1: 0 data baru, total 239
239


In [9]:
def tentukan_kategori(nama):
    n = nama.lower()
    if 'kemeja' in n or 'workshirt' in n:                                 return 'Kemeja'
    if 'polo' in n:                                                       return 'Polo'
    if 'jaket' in n or 'jacket' in n:                                     return 'Jaket'
    if 'hoodie' in n or 'sweater' in n:                                   return 'Hoodie'
    if 'kaos' in n or 't-shirt' in n or 'tshirt' in n or 't shirt' in n:  return 'Kaos'
    return None

hasil = []
for row in result:
    kat = tentukan_kategori(row['nama'])
    if kat is not None:
        row['kategori'] = kat
        hasil.append(row)

print(len(result), '->', len(hasil))

for row in hasil:
    print(row)

with open('hasil_scraping.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=hasil[0].keys())
    w.writeheader()
    for row in hasil:
        w.writerow(row)

239 -> 234
{'nama': 'Johnwin - Kaos Active Fashion Pria motif crew neck sablon URBAN STYLE', 'harga': 'Rp133.000', 'harga_asli': 'Rp283.600', 'rating': '5.0', 'terjual': '3 terjual', 'toko': 'Johnwin', 'lokasi': 'Jakarta Barat', 'produk': 'https://www.tokopedia.com/johnwin/johnwin-kaos-active-fashion-pria-motif-crew-neck-sablon-urban-style-s?extParam=ivf%3Dfalse%26keyword%3Dkaos+pria%26search_id%3D20261107153433CD00692C3BFE5903C07F%26src%3Dsearch', 'kategori': 'Kaos'}
{'nama': 'Cressida Kaos Kerah Regular Pria Multi Colour - PMWMS.RB022X', 'harga': 'Rp254.610', 'harga_asli': 'Rp333.700', 'rating': '5.0', 'terjual': '2 terjual', 'toko': 'Cressida', 'lokasi': 'Bandung', 'produk': 'https://www.tokopedia.com/cressida/cressida-kaos-kerah-regular-men-multi-colour-pmwms-rb022x-1730388349592176468?extParam=ivf%3Dfalse%26keyword%3Dkaos+pria%26search_id%3D20261107153433CD00692C3BFE5903C07F%26src%3Dsearch', 'kategori': 'Kaos'}
{'nama': 'Kaos Distro Cowok ORUKA BULB Baju Pria Sablon Premium T-Shir

In [10]:
df = pd.read_csv('hasil_scraping.csv')
df

,nama,harga,harga_asli,rating,terjual,toko,lokasi,produk,kategori
0,Johnwin - Kaos Active Fashion Pria motif crew ...,Rp133.000,Rp283.600,5.0,3 terjual,Johnwin,Jakarta Barat,https://www.tokopedia.com/johnwin/johnwin-kaos...,Kaos
1,Cressida Kaos Kerah Regular Pria Multi Colour ...,Rp254.610,Rp333.700,5.0,2 terjual,Cressida,Bandung,https://www.tokopedia.com/cressida/cressida-ka...,Kaos
2,Kaos Distro Cowok ORUKA BULB Baju Pria Sablon ...,Rp82.450,Rp85.000,5.0,60+ terjual,Oruka Cloth,Blitar,https://www.tokopedia.com/oruka-official/kaos-...,Kaos
3,Cressida Kaos Kerah Regular Pria Multi Colour ...,Rp273.900,Rp323.700,5.0,4 terjual,Cressida,Bandung,https://www.tokopedia.com/cressida/cressida-ka...,Kaos
4,Kaos Cowok Distro Yaris Baju Kaos T Shirt Pria...,Rp97.350,Rp177.000,5.0,6 terjual,Oruka Cloth,Blitar,https://www.tokopedia.com/oruka-official/kaos-...,Polo
...,...,...,...,...,...,...,...,...,...
229,JAKET JEANS PRIA WANITA HIPHOP BLIT FULL BORDI...,Rp164.999,Rp250.000,4.8,10rb+ terjual,MAMIMA,Jakarta Barat,https://www.tokopedia.com/mamima-852/jaket-jea...,Jaket
230,jaket outdoor keren pria Aquateach jaket paras...,Rp134.999,Rp249.500,4.5,100+ terjual,Rattop,Kab. Bandung,https://www.tokopedia.com/rattop/jaket-outdoor...,Jaket
231,Greenlight Jaket Pria Tracksuit Jacket Regular...,Rp239.000,Rp499.000,4.8,10rb+ terjual,3SECOND,Bandung,https://www.tokopedia.com/3second/greenlight-j...,Jaket
232,Greenlight Essential Jaket Bomber Pria Regular...,Rp254.000,Rp529.000,4.9,3rb+ terjual,GREENLIGHT STORE,Bandung,https://www.tokopedia.com/itsgreenlightofficia...,Jaket


In [11]:
def ubah_terjual(t):
    t = str(t).lower().replace('terjual', '').replace('+', '').strip()
    if t in ('', '-', 'nan'):
        return np.nan
    if 'jt' in t:
        return float(t.replace('jt', '').replace(',', '.').strip()) * 1000000
    if 'rb' in t:
        return float(t.replace('rb', '').replace(',', '.').strip()) * 1000
    return float(re.sub(r'[^0-9.]', '', t))

for k in ['harga', 'harga_asli']:
    df[k] = pd.to_numeric(df[k].astype(str).str.replace(r'[^0-9]', '', regex=True), errors='coerce')
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
df['terjual'] = df['terjual'].apply(ubah_terjual)

df.info()
df.to_csv('hasil_scraping.csv', index=False)
df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 234 entries, 0 to 233
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   nama        234 non-null    object 
 1   harga       234 non-null    int64  
 2   harga_asli  234 non-null    int64  
 3   rating      234 non-null    float64
 4   terjual     234 non-null    float64
 5   toko        234 non-null    object 
 6   lokasi      234 non-null    object 
 7   produk      234 non-null    object 
 8   kategori    234 non-null    object 
dtypes: float64(2), int64(2), object(5)
memory usage: 16.6+ KB


,nama,harga,harga_asli,rating,terjual,toko,lokasi,produk,kategori
0,Johnwin - Kaos Active Fashion Pria motif crew ...,133000,283600,5.0,3.0,Johnwin,Jakarta Barat,https://www.tokopedia.com/johnwin/johnwin-kaos...,Kaos
1,Cressida Kaos Kerah Regular Pria Multi Colour ...,254610,333700,5.0,2.0,Cressida,Bandung,https://www.tokopedia.com/cressida/cressida-ka...,Kaos
2,Kaos Distro Cowok ORUKA BULB Baju Pria Sablon ...,82450,85000,5.0,60.0,Oruka Cloth,Blitar,https://www.tokopedia.com/oruka-official/kaos-...,Kaos
3,Cressida Kaos Kerah Regular Pria Multi Colour ...,273900,323700,5.0,4.0,Cressida,Bandung,https://www.tokopedia.com/cressida/cressida-ka...,Kaos
4,Kaos Cowok Distro Yaris Baju Kaos T Shirt Pria...,97350,177000,5.0,6.0,Oruka Cloth,Blitar,https://www.tokopedia.com/oruka-official/kaos-...,Polo
...,...,...,...,...,...,...,...,...,...
229,JAKET JEANS PRIA WANITA HIPHOP BLIT FULL BORDI...,164999,250000,4.8,10000.0,MAMIMA,Jakarta Barat,https://www.tokopedia.com/mamima-852/jaket-jea...,Jaket
230,jaket outdoor keren pria Aquateach jaket paras...,134999,249500,4.5,100.0,Rattop,Kab. Bandung,https://www.tokopedia.com/rattop/jaket-outdoor...,Jaket
231,Greenlight Jaket Pria Tracksuit Jacket Regular...,239000,499000,4.8,10000.0,3SECOND,Bandung,https://www.tokopedia.com/3second/greenlight-j...,Jaket
232,Greenlight Essential Jaket Bomber Pria Regular...,254000,529000,4.9,3000.0,GREENLIGHT STORE,Bandung,https://www.tokopedia.com/itsgreenlightofficia...,Jaket
